## Make or Buy
In-House Production:
<ul>
<li>Fixed monthly cost (if any legs are produced): \$10,000
<li>Variable cost per leg: \$5
<li>Maximum in-house capacity: 2,000 legs
</ul>
Supplier Option:
<ul>
<li>Cost per leg from supplier: \$8
<li>No capacity limit (the supplier can meet any demand)
<li>Monthly Demand:
</ul>
The company needs 1,500 table legs per month.
<p>
Objective:
<ul>
<li>Minimize total monthly cost.
</ul>

In [2]:
pip install pulp

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ----- ---------------------------------- 2.4/16.4 MB 16.8 MB/s eta 0:00:01
   --------------- ------------------------ 6.6/16.4 MB 16.8 MB/s eta 0:00:01
   ----------------- ---------------------- 7.3/16.4 MB 12.2 MB/s eta 0:00:01
   ---------------------------- ----------- 11.8/16.4 MB 14.7 MB/s eta 0:00:01
   ---------------------------------------  16.3/16.4 MB 16.2 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16.4 MB 15.6 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pulp import LpMaximize, LpProblem, LpStatus, LpVariable, LpMinimize, value

In [4]:
# Set up maximization model
Table_Leg_Model = LpProblem("Make_or_Buy_Decision", LpMinimize)

In [5]:
# Define decision variables
x = LpVariable("InHouse_Production", lowBound=0, upBound=2000, cat="Integer")  # Number of legs produced in-house
y = LpVariable("Supplier_Production", lowBound=0, cat="Integer")              # Number of legs purchased from supplier
z = LpVariable("InHouse_Fixed_Cost", cat="Binary")                            # Binary variable for fixed cost

In [6]:
# Define objective function (minimize total cost)
# Total cost = Variable cost for in-house + Supplier cost + Fixed cost if any in-house production is used
obj_func = 5*x + 8*y + 10000*z
# Add Objective function to the model
Table_Leg_Model += obj_func

In [7]:
# Add constraints to the model
# Demand constraint: Total production (in-house + supplier) should meet demand
Table_Leg_Model += (x + y == 1500, "Demand_Constraint")

# Capacity constraint for in-house production
Table_Leg_Model += (x <= 2000, "InHouse_Capacity")

# Link fixed cost to in-house production: z = 1 if x > 0, else z = 0
# Setting this constraint to ensure z is 1 if x > 0 (any in-house production)
Table_Leg_Model += (x <= 2000 * z, "Fixed_Cost_Trigger")

In [8]:
# Review the model
Table_Leg_Model

Make_or_Buy_Decision:
MINIMIZE
10000*InHouse_Fixed_Cost + 5*InHouse_Production + 8*Supplier_Production + 0
SUBJECT TO
Demand_Constraint: InHouse_Production + Supplier_Production = 1500

InHouse_Capacity: InHouse_Production <= 2000

Fixed_Cost_Trigger: - 2000 InHouse_Fixed_Cost + InHouse_Production <= 0

VARIABLES
0 <= InHouse_Fixed_Cost <= 1 Integer
0 <= InHouse_Production <= 2000 Integer
0 <= Supplier_Production Integer

In [9]:
status = Table_Leg_Model.solve()
print ("status returned code :", status)
print ({0: 'Not Solved', 1: 'Optimal', -1: 'Infeasible', -2: 'Unbounded', -3: 'Undefined'})


status returned code : 1
{0: 'Not Solved', 1: 'Optimal', -1: 'Infeasible', -2: 'Unbounded', -3: 'Undefined'}


In [10]:
print("Solution is", LpStatus[Table_Leg_Model.status])
print("objective :", Table_Leg_Model.objective.value())

for var in Table_Leg_Model.variables():
    print(var.name, ":", var.value())
for name, constraint in Table_Leg_Model.constraints.items():
    print(name, " : ", constraint.value())

Solution is Optimal
objective : 12000.0
InHouse_Fixed_Cost : 0.0
InHouse_Production : 0.0
Supplier_Production : 1500.0
Demand_Constraint  :  0.0
InHouse_Capacity  :  -2000.0
Fixed_Cost_Trigger  :  0.0


In [11]:
# using value() from pulp to retrieve results
print("Status:", LpStatus[Table_Leg_Model.status])
print("In-House Production (units):", value(x))
print("Supplier Production (units):", value(y))
print("Total Monthly Cost ($):", value(Table_Leg_Model.objective))

Status: Optimal
In-House Production (units): 0.0
Supplier Production (units): 1500.0
Total Monthly Cost ($): 12000.0
